# D5.2 · The root cause record

**Function D — The Agentic SOC → The Agentic SOC — Recover and Root Cause**  ·  *Security of AI*

Builds on **[D5.1 · Replay and forensics](https://spbreed.github.io/cyber-commons/lessons/D5.1.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** The root cause record: walking the control chain, marking each control present, absent or present-but-wrong, and testing whether the resulting statement names a control rather than a person.

**Why a security engineer needs it.** Incidents that close with a narrative recur, because nothing in a narrative can be built or measured. Naming the control makes the finding actionable and gives the next stage something to verify. The test is mechanical on purpose — "the engineer missed the alert" is true, and it is not a root cause.

## 1 · The hook

The postmortem says the on-call engineer missed the alert. It is true, it is useless, and it will be true again next quarter about somebody else. A root cause names a control, or it names nothing that can be built.

> **At CyberTravels.** The control chain is CyberTravels': provenance at ingress (A2.6), default-deny on the tool call (A3.1), the detection that should have caught a refund without an approval, and the stop authority that existed and was never reached. The first absent one is the root cause; the stop lever nobody pulled is evidence about the detection in front of it.

## 2 · The framework

```
   control chain for INC-2026-114        status

   A2.6  provenance at ingress           absent   <- first absent = ROOT
   A3.1  default-deny on the tool call   wrong scope
   D2.1  detection: refund w/o approval  absent   <- contributing
   D1.3  drift on vendor tool descs      absent   <- contributing
   D4.4  stop authority in the window    PRESENT  <- and never reached

   a control that exists but is never reached is not a mitigating factor.
   it is evidence about the detection in front of it.

   statement test:  names a control -> usable
                    names a person  -> rejected, however true
```

"The on-call engineer missed the alert" is true and useless — it will happen
again next quarter to a different engineer. "We should have been more careful"
names no control and no change anyone can make.

A usable root cause names **the control that should have caught this and did
not**, in a form the next change can act on and the KCIs can later measure. The
test is mechanical, and it should be: does the statement name a control?

## 3 · Present, absent, or present-but-wrong

Three categories, and the third is the one that gets missed.

In the CyberTravels incident, stop authority was **present** and the incident
still ran 194 minutes. A control that exists but is never reached is not a
mitigating factor — it is evidence that the detection in front of it was the
gap, which is a different finding and a different fix.

## 4 · Three candidate statements, one usable

The rejections are the lesson. Both rejected statements are true; neither can be built, tested or measured.

### The skill — [`skills/response/root-cause-record/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/root-cause-record/SKILL.md)

```yaml
name: root-cause-record
description: >-
  Turn a reconstructed incident into a root cause record that names the control
  that failed, and reject statements that name a person or an intention. Use when
  closing an incident, when a postmortem produces a narrative instead of a
  change, or when the same incident class keeps recurring.
allowed-tools: Read, Grep, Glob
```

# A root cause is a control, not a story and not a person

"The on-call engineer missed the alert" is true and useless: it will happen
again next quarter to a different engineer. "We should have been more careful"
names no control and no change.

A usable root cause names **the control that should have caught this and did
not**, in a form the next change can act on and `kci-fix-validation` can later
measure.

## When to use this

At incident close, before the postmortem is written, and in review of any
incident whose remediation was "increased awareness" or "additional training".

## Step-by-step

**1 — List every control in the chain, not just the one that failed last.**
Detection, prevention, authority to stop.

**2 — Mark each present, absent, or present-but-wrong.** The third category is
the one that gets missed.

**3 — The first absent control in the chain is the root cause candidate.**
Later absences are contributing.

**4 — Test the statement: does it name a control?** If it names a person, a
team, or a state of mind, it is rejected. This is mechanical and should be.

**5 — Carry the named control forward as a KCI.** That is the handover to
D5.3 and to the policy change proposal.

## Example

**Input** — a five-control chain and three candidate statements, in
[`scripts/root_cause_record.py`](scripts/root_cause_record.py).

**Output** — a real run:

```
   REJECT  the on-call engineer missed the alert
           -> names no control
   ACCEPT  no control compared the vendor tool description against the version approved at onboarding
```

## Output contract

```json
{
  "incident": "str",
  "root_control": "str",
  "contributing": ["str"],
  "statements": [{"text": "str", "accepted": true}]
}
```

## Common edge cases

- **The control existed and was never reached.** That is evidence about the
  detection in front of it, not a mitigating factor.
- **Several controls absent at once.** The first in the chain is root; naming
  all of them as root produces a list nobody actions.
- **A genuine human error with no control behind it.** Then the finding is that
  no control existed — which is a control statement.

## Failure modes

- **Narrative postmortems.** Readable, and they change nothing.
- **Blame, however gently worded.** It ends the analysis one step early.
- **A root cause with no KCI.** Nothing will ever check that the fix worked.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/root-cause-record/scripts/root_cause_record.py
SCRIPT = "skills/response/root-cause-record/scripts/root_cause_record.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

One of three statements accepted. The first absent control in the chain is the root cause; the later absences are contributing.

## Your turn

Take your last postmortem's root cause and run the test on it. If it names a person or an intention, rewrite it as a control.

---

**Next → [D5.3 · Validating the fix against the KCIs](https://spbreed.github.io/cyber-commons/lessons/D5.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D5.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D5.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*